https://analyse.kmi.open.ac.uk/open-dataset	

In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [21]:
#SECTION TO IMPORT CSV FILES FROM THE OULAD DATA SET (EDIT FILE LOCATIONS IF THEY ARE NOT ON THE SAME LEVEL AS THIS PYTHON SCRIPT)
print("Importing data...")
results = pd.read_csv("studentInfo.csv")
vle = pd.read_csv("studentVle.csv")
grades = pd.read_csv("studentAssessment.csv")
assessments = pd.read_csv("assessments.csv")


Importing data...


In [25]:
print(results.shape) #(32593, 14)
print(vle.shape) #(10655280, 6)
print(grades.shape) #(173912, 11)
print(assessments.shape) #(206, 6)

(32593, 14)
(10655280, 6)
(173912, 11)
(206, 6)


In [22]:
#calculation of "average_score"
print("Preparing data...")
grades = grades.merge(assessments,on="id_assessment",how="left")
grades["weighted_score"] = grades['score']*grades["weight"]/100.0
av_mark = grades.groupby("id_student")["weighted_score"].sum()/(grades.groupby("id_student")["weight"].sum()/100)
results = results.join(av_mark.rename("average_score"),on=["id_student"],how="left") #merge "average_score" into results
results["average_score"]=results["average_score"].fillna(results["average_score"].mean()) #fill in missing data with the mean

#calculation of "sum_clicks"
studentClicksPerModule = vle.groupby(["code_module","code_presentation","id_student"])["sum_click"].count()
results = results.join(studentClicksPerModule,on=["code_module","code_presentation","id_student"],how='left')

#the removal of our unique identifier ["code_module","code_presentation","id_student"], now that all needed data has been merged to results, and cleaning of the unwanted "region" field
ids = results[["code_module","code_presentation","id_student"]]
X0 = results.drop(["code_module","code_presentation","id_student","region"], axis=1)
print(data.shape)

Preparing data...
(19362, 13)


In [23]:
#integer encoding and filling of missing data with mean
X0["gender"] = X0["gender"].replace(["M","F"],[0,1])
X0["disability"] = X0["disability"].replace(["N","Y"],[1,0])
X0["age_band"] = X0["age_band"].replace(["0-35","35-55","55<="],[1,2,3])
X0["imd_band"] = X0["imd_band"].replace(["Oct-20","10-20","0-10%","10-20%","20-30%","30-40%","40-50%","50-60%","60-70%","70-80%","80-90%","90-100%"],[np.nan,np.nan,0.05,0.15,0.25,0.35,0.45,0.55,0.65,0.75,0.85,0.95])
# X0["imd_band"] = X0["imd_band"].fillna(X0["imd_band"].mean())
X0["highest_education"] = X0["highest_education"].replace(["No Formal quals","Lower Than A Level","A Level or Equivalent","HE Qualification","Post Graduate Qualification"],[0,1,2,3,4])
# X0["sum_click"]=X0["sum_click"].fillna(X0["sum_click"].mean())
X0.dropna(inplace=True)
# #One-Hot-Encoding section
X1 = X0.drop(["highest_education","imd_band","sum_click","average_score","num_of_prev_attempts", "final_result","studied_credits"], axis=1)
X0 = X0.drop(["age_band","disability","gender"], axis=1)
encoder = OneHotEncoder(handle_unknown='ignore')
X0 = X0.join(pd.DataFrame(encoder.fit_transform(X1).toarray()))

X0.dropna(inplace=True)
X0 = X0.drop("final_result",axis=1)
X0['score'] = X0['average_score']
X0 = X0.drop("average_score",axis=1)

C:\Users\Zongwen\AppData\Local\Temp\ipykernel_10972\3988915830.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X0["gender"] = X0["gender"].replace(["M","F"],[0,1])
C:\Users\Zongwen\AppData\Local\Temp\ipykernel_10972\3988915830.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X0["disability"] = X0["disability"].replace(["N","Y"],[1,0])
C:\Users\Zongwen\AppData\Local\Temp\ipykernel_10972\3988915830.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old b

# Final data used for model evaluation

In [24]:
data = X0.values
print(data.shape)

(19362, 13)
